# Least Squares and Residuals

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 20)

## Review: The Regression Estimate

Last class we found the regression line. In standard units,

$$\text{estimate of } y = r \times x$$

A trivia tournament has a first round (average score 70, SD 10) and a much harder final round (average 50, SD 12). The scatter plot of the two scores is oval shaped, with $r = 0.75$. What do we expect the average final round score to be for players who scored 90 in the first round?

In [ ]:
average_x <- 70
sd_x <- 10
average_y <- 50
sd_y <- 12
r <- 0.75

Step 1: convert 90 to standard units.

In [ ]:
x_su <- (90 - average_x) / sd_x
x_su

Step 2: multiply by $r$. That's the estimate of $y$ in standard units.

In [ ]:
y_su <- r * x_su
y_su

Step 3: convert back to the original units of $y$.

In [ ]:
y_su * sd_y + average_y

The slope and intercept give the same estimate in one step.

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}$$

$$\text{intercept} = \text{average of } y - \text{slope} \times \text{average of } x$$

In [ ]:
slope <- r * sd_y / sd_x
intercept <- average_y - slope * average_x
c(slope, intercept)

In [ ]:
slope * 90 + intercept

What about players who scored 60 in the first round?

In [ ]:
slope * 60 + intercept

## Part 1: Error in Estimation

`district_demographics2016.csv` has one row for each of the 435 U.S. congressional districts in 2016. `Median Income` is the district's median income in dollars. `College%` is the percent of people in the district who graduated from college.

In [ ]:
demographics <- read_csv("data/district_demographics2016.csv", show_col_types = FALSE)
demographics

We'll predict `Median Income` from `College%`, so we keep those two columns.

In [ ]:
demographics <- demographics |>
  select(`Median Income`, `College%`)

demographics

In [ ]:
ggplot(demographics, aes(x = `College%`, y = `Median Income`)) +
  geom_point(alpha = 0.5) +
  theme_minimal(base_size = 16)

Before you run the next cell: what's your guess for $r$?

In [ ]:
demographics_r <- cor(demographics$`College%`, demographics$`Median Income`)
demographics_r

The slope and intercept of the regression line, from the formulas:

In [ ]:
demographics_slope <- demographics_r * sd(demographics$`Median Income`) / sd(demographics$`College%`)
demographics_intercept <- mean(demographics$`Median Income`) - demographics_slope * mean(demographics$`College%`)
c(demographics_slope, demographics_intercept)

`lm()` fits the same line.

In [ ]:
demographics_model <- lm(`Median Income` ~ `College%`, data = demographics)
coef(demographics_model)

`fitted()` gives the line's estimate for every row. The gold points are those estimates.

In [ ]:
demographics <- demographics |>
  mutate(`Linear Prediction` = fitted(demographics_model))

ggplot(demographics, aes(x = `College%`)) +
  geom_point(aes(y = `Median Income`), alpha = 0.5) +
  geom_point(aes(y = `Linear Prediction`), color = "goldenrod", size = 2) +
  labs(y = "Median Income") +
  theme_minimal(base_size = 16)

Each district has an **error**: the actual value minus the estimate. It's the vertical distance between the point and the line, and it's negative if the point is below the line.

$$\text{error} = \text{actual value} - \text{estimate}$$

In [ ]:
demographics <- demographics |>
  mutate(Error = `Median Income` - `Linear Prediction`)

demographics

How big are the errors overall? Try the average.

In [ ]:
mean(demographics$Error)

That's 0, with a tiny rounding error. The positive and negative errors cancel. So square the errors first, take the mean, and then take the square root to get back to dollars. This is the **root mean squared error (rmse)**.

In [ ]:
sqrt(mean(demographics$Error^2))

### Other Lines

`demographics_errors()` takes any slope and any intercept (in that order) and draws that line. The red segments are the errors for four of the districts. There's nothing special about those four. They were chosen so the picture is easy to read.

In [ ]:
sample_points <- tibble(
  x = c(14.7, 19.1, 50.7, 59.5),
  y = c(33995, 61454, 71183, 105918)
)

demographics_errors <- function(slope, intercept) {
  ggplot(demographics, aes(x = `College%`, y = `Median Income`)) +
    geom_point(alpha = 0.5) +
    geom_abline(slope = slope, intercept = intercept,
                color = "dodgerblue3", linewidth = 1.2) +
    geom_segment(data = sample_points,
                 aes(x = x, xend = x, y = y, yend = slope * x + intercept),
                 color = "red", linewidth = 1.2) +
    theme_minimal(base_size = 16)
}

The regression line:

In [ ]:
demographics_errors(demographics_slope, demographics_intercept)

A different line. Which errors got smaller? Which got bigger?

In [ ]:
demographics_errors(1500, 20000)

And a silly one.

In [ ]:
demographics_errors(-1000, 75000)

### Root Mean Squared Error

`demographics_rmse()` draws the same picture and prints the rmse of the line.

In [ ]:
demographics_rmse <- function(slope, intercept) {
  x <- demographics$`College%`
  y <- demographics$`Median Income`
  fitted <- slope * x + intercept
  mse <- mean((y - fitted)^2)
  cat("Root mean squared error:", sqrt(mse), "\n")
  demographics_errors(slope, intercept)
}

In [ ]:
demographics_rmse(-1000, 75000)

In [ ]:
demographics_rmse(1500, 20000)

In [ ]:
demographics_rmse(demographics_slope, demographics_intercept)

The regression line has the smallest rmse of the three. Is there a line that does better?

## Part 2: Numerical Optimization

`optim()` finds the input that makes a function's output as small as possible. It searches by trial and error, moving toward lower outputs. Start with a function where we can see the answer.

In [ ]:
f <- function(x) {
  (x - 2)^2 + 3
}

In [ ]:
tibble(x = seq(1, 3, by = 0.1), y = f(x)) |>
  ggplot(aes(x, y)) +
  geom_line(color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

`optim()` needs a starting value for the search (here 0) and the function. `$par` is the input it found. Its default search needs at least two numbers to adjust, so for a function of one number we ask for a different search with `method = "BFGS"`.

In [ ]:
optim(0, f, method = "BFGS")$par

It also works when we can't see the answer from the formula.

In [ ]:
complicated_function <- function(x) {
  2 * sin(x * pi) + x^3 + x^4
}

In [ ]:
tibble(x = seq(-1.5, 1.5, by = 0.05), y = complicated_function(x)) |>
  ggplot(aes(x, y)) +
  geom_line(color = "dodgerblue3", linewidth = 1.2) +
  theme_minimal(base_size = 16)

In [ ]:
optim(0, complicated_function, method = "BFGS")$par

### Minimizing the Mean Squared Error

The line that minimizes the rmse also minimizes the mean squared error (mse). The square root makes no difference to which line wins, so we skip it.

`demographics_mse()` returns the mse of any line. `optim()` needs all the numbers it can change in one vector, so the function takes `params`, where `params[1]` is the slope and `params[2]` is the intercept.

In [ ]:
demographics_mse <- function(params) {
  any_slope <- params[1]
  any_intercept <- params[2]
  x <- demographics$`College%`
  y <- demographics$`Median Income`
  fitted <- any_slope * x + any_intercept
  mean((y - fitted)^2)
}

Check it against the lines we tried. Take the square root to compare with the rmse values above.

In [ ]:
sqrt(demographics_mse(c(1500, 20000)))

In [ ]:
sqrt(demographics_mse(c(-1000, 75000)))

Now let `optim()` search over all lines, starting from a slope of 0 and an intercept of 0.

In [ ]:
best <- optim(c(0, 0), demographics_mse)$par
best

Compare with the slope and intercept from the formulas.

In [ ]:
c(demographics_slope, demographics_intercept)

They match, up to the small inexactness of the search. The regression line minimizes the mean squared error among all lines. That's why it's also called the **least squares line**.

## Part 3: Nonlinear Regression

A 2013 study looked at 28 female collegiate shot put athletes. `Weight Lifted` is the most the athlete lifted in the "1RM power clean" in the pre-season, in kilograms. `Shot Put Distance` is the athlete's personal best, in meters.

In [ ]:
shotput <- read_csv("data/shotput.csv", show_col_types = FALSE)
shotput

In [ ]:
ggplot(shotput, aes(x = `Weight Lifted`, y = `Shot Put Distance`)) +
  geom_point() +
  theme_minimal(base_size = 16)

That's not football shaped, and it looks a little curved. There is still one best straight line. The formulas (through `lm()`) and `optim()` agree on it.

In [ ]:
shotput_model <- lm(`Shot Put Distance` ~ `Weight Lifted`, data = shotput)
coef(shotput_model)

In [ ]:
shotput_linear_mse <- function(params) {
  any_slope <- params[1]
  any_intercept <- params[2]
  x <- shotput$`Weight Lifted`
  y <- shotput$`Shot Put Distance`
  fitted <- any_slope * x + any_intercept
  mean((y - fitted)^2)
}

optim(c(0, 0), shotput_linear_mse)$par

In [ ]:
shotput |>
  mutate(`Best Straight Line` = fitted(shotput_model)) |>
  ggplot(aes(x = `Weight Lifted`)) +
  geom_point(aes(y = `Shot Put Distance`)) +
  geom_point(aes(y = `Best Straight Line`), color = "goldenrod", size = 2) +
  labs(y = "Shot Put Distance") +
  theme_minimal(base_size = 16)

The line is too high at both ends and too low in the middle. A curve would fit better. The study suggested a quadratic function:

$$f(x) = ax^2 + bx + c$$

Least squares works the same way. Write a function that returns the mse for any $a$, $b$, and $c$, and let `optim()` find the best three numbers.

In [ ]:
shotput_quadratic_mse <- function(params) {
  a <- params[1]
  b <- params[2]
  c <- params[3]
  x <- shotput$`Weight Lifted`
  y <- shotput$`Shot Put Distance`
  fitted <- a * x^2 + b * x + c
  mean((y - fitted)^2)
}

In [ ]:
best <- optim(c(0, 0, 0), shotput_quadratic_mse)$par
best

The predicted distance for an athlete who lifts 100 kilograms:

In [ ]:
best[1] * 100^2 + best[2] * 100 + best[3]

In [ ]:
shotput |>
  mutate(`Best Quadratic Curve` = best[1] * `Weight Lifted`^2 +
           best[2] * `Weight Lifted` + best[3]) |>
  ggplot(aes(x = `Weight Lifted`)) +
  geom_point(aes(y = `Shot Put Distance`)) +
  geom_point(aes(y = `Best Quadratic Curve`), color = "goldenrod", size = 2) +
  labs(y = "Shot Put Distance") +
  theme_minimal(base_size = 16)

## Part 4: Residuals

The errors from the regression line have their own name: **residuals**.

$$\text{residual} = \text{observed value} - \text{regression estimate}$$

Back to the heights from last class. `fitted()` gives the regression estimates and `residuals()` gives the residuals of a model made by `lm()`.

In [ ]:
original <- read_csv("data/family_heights.csv", show_col_types = FALSE)

heights <- tibble(
  MidParent = original$midparentHeight,
  Child = original$childHeight
)

family_model <- lm(Child ~ MidParent, data = heights)
coef(family_model)

In [ ]:
heights <- heights |>
  mutate(
    `Fitted Value` = fitted(family_model),
    Residual = residuals(family_model)
  )

heights

The average of the residuals is 0, the same as the errors in Part 1.

In [ ]:
mean(heights$Residual)

`scatter_fit()` draws the scatter plot with the regression line. It takes the table and the names of the $x$ and $y$ columns in quotes.

In [ ]:
scatter_fit <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  ggplot(table, aes(x = .data[[x]], y = .data[[y]])) +
    geom_point(alpha = 0.5) +
    geom_abline(intercept = coef(model)[1], slope = coef(model)[2],
                color = "gold", linewidth = 2) +
    labs(x = x, y = y) +
    theme_minimal(base_size = 16)
}

In [ ]:
scatter_fit(heights, "MidParent", "Child")

A **residual plot** has the residuals on the vertical axis and $x$ on the horizontal axis. `residual_plot()` fits the regression with `lm()` and plots the residuals.

In [ ]:
residual_plot <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  tibble(x_values = table[[x]], residuals = residuals(model)) |>
    ggplot(aes(x = x_values, y = residuals)) +
    geom_point(color = "red", alpha = 0.5) +
    geom_hline(yintercept = 0, color = "darkblue", linewidth = 2) +
    labs(x = x, title = "Residual Plot") +
    theme_minimal(base_size = 16)
}

In [ ]:
residual_plot(heights, "MidParent", "Child")

No pattern: the residuals look about the same above and below 0, all the way across. That's what the residual plot of a good regression looks like.

`regression_diagnostic_plots()` draws both plots. A function only returns its last value, so it uses `print()` to show both.

In [ ]:
regression_diagnostic_plots <- function(table, x, y) {
  print(scatter_fit(table, x, y))
  print(residual_plot(table, x, y))
}

### Detecting Nonlinearity

Dugongs are marine mammals related to manatees. `Age` is in years and `Length` is in meters. If we measure a dugong's length, what can we say about its age?

In [ ]:
dugong <- read_csv("data/dugongs.csv", show_col_types = FALSE) |>
  select(Length, Age)

dugong

In [ ]:
cor(dugong$Length, dugong$Age)

In [ ]:
regression_diagnostic_plots(dugong, "Length", "Age")

The correlation is high, but the residuals are positive, then negative, then positive again. The line's estimates are too low, then too high, then too low. A curve would do better.

**When a residual plot shows a pattern, there may be a nonlinear relation between the variables.**

### Detecting Uneven Spread

Back to the districts from Part 1.

In [ ]:
regression_diagnostic_plots(demographics, "College%", "Median Income")

The residual plot fans out to the right. The errors are bigger for districts with more college graduates.

**If the residual plot shows uneven spread around the line at 0, the regression estimates are not equally accurate across the range of $x$.**

## Try it out!

Last class you fit a line to predict a penguin's body mass (`body_mass_g`, in grams) from its flipper length (`flipper_length_mm`). How well does it fit?

1. Add two columns to `model_data`: the fitted values and the residuals.
2. Find the average of the residuals.
3. Find the rmse. What are its units?
4. Draw the residual plot with `residual_plot()`. Do you see a pattern?

**Done early?** Which penguin has the largest residual?

In [ ]:
library(palmerpenguins)

# Keep rows with the two variables we need
model_data <- penguins |>
  filter(!is.na(flipper_length_mm), !is.na(body_mass_g)) |>
  select(species, flipper_length_mm, body_mass_g)

penguin_model <- lm(body_mass_g ~ flipper_length_mm, data = model_data)
coef(penguin_model)

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Done early? The largest residual